# Single-frame Object Replace (GroundingDINO + SAM2 + Quantized FLUX Fill)

Ноутбук self-contained и рассчитан на Colab/CUDA.

Что исправлено:
- убраны конфликтующие install-ячейки;
- зафиксирована совместимая версия `transformers==4.47.1` для `groundingdino-py`;
- SAM2 загружается через `SAM2ImagePredictor.from_pretrained(...)`;
- FLUX Fill грузится в квантизованном режиме (4-bit с fallback на 8-bit);
- вызов pipeline адаптивный к сигнатуре (чтобы не падать на `negative_prompt`);
- добавлены memory-оптимизации и ограничение размера ROI.


In [ ]:
# @title 0) Install (run once), then Runtime -> Restart runtime

%pip -q install -U pip setuptools wheel
%pip -q install -U   diffusers==0.35.1   transformers==4.47.1   accelerate>=0.34,<1   safetensors>=0.4.3   huggingface_hub>=0.26,<1   groundingdino-py==0.4.0   opencv-python   matplotlib   scipy   bitsandbytes

%pip -q install -U git+https://github.com/facebookresearch/sam2.git


In [1]:
# @title 1) Imports + runtime config
import os
import inspect
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

import diffusers
import transformers
import huggingface_hub

from huggingface_hub import login, hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

from diffusers import FluxFillPipeline, FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig

from transformers import T5EncoderModel
from transformers import BitsAndBytesConfig as TransformersBitsAndBytesConfig

from groundingdino.util.inference import load_model, predict
from sam2.sam2_image_predictor import SAM2ImagePredictor

import torchvision.transforms as T


def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = pick_device()
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("torch:", torch.__version__)
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("device:", DEVICE, "| dtype:", DTYPE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

major_minor = tuple(int(x) for x in transformers.__version__.split(".")[:2])
assert major_minor < (4, 48), "Need transformers<4.48 for groundingdino-py. Re-run install cell and restart runtime."


/Users/dalerkhodzhimatov/miniconda3/envs/venv/lib/python3.10/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/Users/dalerkhodzhimatov/miniconda3/envs/venv/lib/python3.10/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/Users/dalerkhodzhimatov/miniconda3/envs/venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


torch: 2.10.0
diffusers: 0.36.0
transformers: 4.47.1
huggingface_hub: 0.36.2
device: mps | dtype: torch.float32


In [ ]:
# @title 2) Hugging Face auth (required for gated FLUX Fill model)
# Put your HF token (Read access). Also accept model license on model page.
HF_TOKEN = os.getenv("HF_TOKEN") # @param {type:"string"}

if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)
    print("HF login done")
else:
    print("HF_TOKEN is empty. If you get 401 for FLUX model, set token here.")


HF login done


In [3]:
# @title 3) User settings
IMAGE_PATH = "videos/frame.png"  # @param {type:"string"}

OLD_OBJECT_PROMPT = "clock watck"  # @param {type:"string"}
NEW_OBJECT_PROMPT = "realistic ceramic mug with small alpha text"  # @param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, duplicated object, artifacts"  # @param {type:"string"}

# Must be FLUX Fill model for inpainting/fill task
FLUX_FILL_MODEL_ID = "black-forest-labs/FLUX.1-schnell"  # @param {type:"string"}

TRY_4BIT_FIRST = True  # @param {type:"boolean"}
USE_CPU_OFFLOAD = True  # @param {type:"boolean"}

BOX_THRESHOLD = 0.82  # @param {type:"number"}
TEXT_THRESHOLD = 0.77  # @param {type:"number"}
ROI_PADDING = 48  # @param {type:"integer"}
MASK_DILATE = 9  # @param {type:"integer"}
MASK_FEATHER = 11  # @param {type:"integer"}
NUM_STEPS = 18  # @param {type:"integer"}
GUIDANCE_SCALE = 14.0  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
MAX_ROI_SIDE = 896  # @param {type:"integer"}

OUT_DIR = "videos/"  # @param {type:"string"}

assert Path(IMAGE_PATH).exists(), f"Image not found: {IMAGE_PATH}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)


In [4]:
# @title 4) Utility functions

def show_image(img, title="", figsize=(7, 7), cmap=None):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()


def preprocess_for_gdino(image_np):
    tr = T.Compose([
        T.Resize((800, 800)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return tr(Image.fromarray(image_np))


def detect_best_bbox(gdino_model, image_np, text_prompt, box_thr=0.3, text_thr=0.25, device="cpu"):
    h, w = image_np.shape[:2]
    image_tensor = preprocess_for_gdino(image_np)

    boxes, logits, phrases = predict(
        model=gdino_model,
        image=image_tensor,
        caption=text_prompt,
        box_threshold=box_thr,
        text_threshold=text_thr,
        device=device,
    )
    if len(boxes) == 0:
        return None

    best = int(logits.argmax())
    cx, cy, bw, bh = boxes[best].tolist()
    x1 = max(0, int((cx - bw / 2) * w))
    y1 = max(0, int((cy - bh / 2) * h))
    x2 = min(w, int((cx + bw / 2) * w))
    y2 = min(h, int((cy + bh / 2) * h))
    return (x1, y1, x2, y2), phrases[best], float(logits[best])


def to_mask255(mask):
    m = mask.astype(np.uint8)
    if m.max() <= 1:
        m = m * 255
    return m


def refine_mask(mask, dilate_px=9, feather_px=11):
    hard = to_mask255(mask)
    if dilate_px > 0:
        k = np.ones((dilate_px, dilate_px), np.uint8)
        hard = cv2.dilate(hard, k, iterations=1)

    soft = hard.copy()
    if feather_px > 0:
        ks = feather_px * 2 + 1
        soft = cv2.GaussianBlur(soft, (ks, ks), 0)
    return hard, soft


def mask_bbox(mask255):
    ys, xs = np.where(mask255 > 0)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def expand_bbox(bbox, w, h, pad=48):
    x1, y1, x2, y2 = bbox
    return max(0, x1 - pad), max(0, y1 - pad), min(w, x2 + pad), min(h, y2 + pad)


def snap_to_multiple(v, base=16, min_v=256):
    return max(min_v, int(np.ceil(v / base) * base))


def clamp_size_keep_aspect(w, h, max_side=896):
    mx = max(w, h)
    if mx <= max_side:
        return w, h
    scale = max_side / float(mx)
    return int(round(w * scale)), int(round(h * scale))


def alpha_insert(base_img, roi_gen, roi_soft_mask, x1, y1, x2, y2):
    out = base_img.copy().astype(np.float32)
    alpha = (roi_soft_mask.astype(np.float32) / 255.0)[..., None]
    src = roi_gen.astype(np.float32)
    dst = out[y1:y2, x1:x2]
    out[y1:y2, x1:x2] = src * alpha + dst * (1.0 - alpha)
    return np.clip(out, 0, 255).astype(np.uint8)


def poisson_insert(base_img, roi_gen, roi_hard_mask, x1, y1, x2, y2):
    src_full = base_img.copy()
    src_full[y1:y2, x1:x2] = roi_gen
    mask_full = np.zeros(base_img.shape[:2], dtype=np.uint8)
    mask_full[y1:y2, x1:x2] = roi_hard_mask

    src_bgr = cv2.cvtColor(src_full, cv2.COLOR_RGB2BGR)
    dst_bgr = cv2.cvtColor(base_img, cv2.COLOR_RGB2BGR)
    center = ((x1 + x2) // 2, (y1 + y2) // 2)
    out_bgr = cv2.seamlessClone(src_bgr, dst_bgr, mask_full, center, cv2.NORMAL_CLONE)
    return cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)


In [5]:
# @title 5) Load GroundingDINO + SAM2
import groundingdino

# GroundingDINO
gdino_ckpt = hf_hub_download(repo_id="ShilongLiu/GroundingDINO", filename="groundingdino_swint_ogc.pth")
gdino_root = Path(groundingdino.__file__).resolve().parent
gdino_cfg = gdino_root / "config" / "GroundingDINO_SwinT_OGC.py"
assert gdino_cfg.exists(), f"GroundingDINO config not found: {gdino_cfg}"

det_seg_device = "cuda" if DEVICE == "cuda" else DEVICE
gdino_model = load_model(str(gdino_cfg), str(gdino_ckpt), device=det_seg_device)
print("GroundingDINO loaded on", det_seg_device)

# SAM2 (no hydra config path issues)
sam2_predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2-hiera-large", device=DEVICE)
print("SAM2 loaded from pretrained")


/Users/dalerkhodzhimatov/miniconda3/envs/venv/lib/python3.10/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased
GroundingDINO loaded on mps
SAM2 loaded from pretrained


In [ ]:
!pip install -U bitsandbytes

In [ ]:
!pip install protobuf

In [ ]:
!pip install sentencepiece
!pip install accelerate

In [ ]:
# @title 6) Load quantized FLUX Fill (4-bit with 8-bit fallback)

def build_flux_fill_auto(model_id: str, prefer_4bit: bool = True):
    import torch

    has_cuda = torch.cuda.is_available()
    has_mps = torch.backends.mps.is_available()

    if has_cuda:
        device = "cuda"
        dtype = torch.bfloat16
    elif has_mps:
        device = "mps"
        dtype = torch.float32
    else:
        device = "cpu"
        dtype = torch.float32

    print(f"Using device: {device}")

    # ================= CUDA PATH (with quantization) =================
    if has_cuda:
        modes = ["4bit", "8bit"] if prefer_4bit else ["8bit", "4bit"]
        last_err = None

        for mode in modes:
            try:
                if mode == "4bit":
                    t_cfg = TransformersBitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_quant_type="nf4",
                        bnb_4bit_use_double_quant=True,
                        bnb_4bit_compute_dtype=dtype,
                    )
                    d_cfg = DiffusersBitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_quant_type="nf4",
                        bnb_4bit_use_double_quant=True,
                        bnb_4bit_compute_dtype=dtype,
                    )
                else:
                    t_cfg = TransformersBitsAndBytesConfig(load_in_8bit=True)
                    d_cfg = DiffusersBitsAndBytesConfig(load_in_8bit=True)

                text_encoder_2 = T5EncoderModel.from_pretrained(
                    model_id,
                    subfolder="text_encoder_2",
                    quantization_config=t_cfg,
                    torch_dtype=dtype,
                )

                transformer = FluxTransformer2DModel.from_pretrained(
                    model_id,
                    subfolder="transformer",
                    quantization_config=d_cfg,
                    torch_dtype=dtype,
                )

                pipe = FluxFillPipeline.from_pretrained(
                    model_id,
                    transformer=transformer,
                    text_encoder_2=text_encoder_2,
                    torch_dtype=dtype,
                )

                pipe.to(device)
                pipe.enable_attention_slicing()

                if getattr(pipe, "vae", None) is not None:
                    pipe.vae.enable_slicing()
                    pipe.vae.enable_tiling()

                print(f"Loaded FLUX Fill in {mode} mode (CUDA)")
                return pipe, mode

            except Exception as e:
                last_err = e
                print(f"[{mode}] failed: {repr(e)}")
                torch.cuda.empty_cache()

        raise RuntimeError(f"CUDA quantized load failed. Last error: {last_err}")

    # ================= CPU / MPS PATH (NO quantization) =================
    else:
        text_encoder_2 = T5EncoderModel.from_pretrained(
            model_id,
            subfolder="text_encoder_2",
            torch_dtype=dtype,
        )

        transformer = FluxTransformer2DModel.from_pretrained(
            model_id,
            subfolder="transformer",
            torch_dtype=dtype,
        )

        pipe = FluxFillPipeline.from_pretrained(
            model_id,
            transformer=transformer,
            text_encoder_2=text_encoder_2,
            torch_dtype=dtype,
        )

        if device == "mps":
            pipe.enable_model_cpu_offload()
        pipe.to(device)
        pipe.enable_attention_slicing()

        if getattr(pipe, "vae", None) is not None:
            pipe.vae.enable_slicing()
            pipe.vae.enable_tiling()

        print("Loaded FLUX Fill without quantization (CPU/MPS)")
        return pipe, "no_quant"


pipe, quant_mode = build_flux_fill_auto(FLUX_FILL_MODEL_ID, prefer_4bit=TRY_4BIT_FIRST)


Using device: mps


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# @title 7) Detect object and get mask
image_np = np.array(Image.open(IMAGE_PATH).convert("RGB"))
h, w = image_np.shape[:2]

det = detect_best_bbox(
    gdino_model,
    image_np,
    OLD_OBJECT_PROMPT,
    box_thr=BOX_THRESHOLD,
    text_thr=TEXT_THRESHOLD,
    device=det_seg_device,
)
if det is None:
    raise RuntimeError("Object not found. Try lower thresholds or another OLD_OBJECT_PROMPT.")

bbox, phrase, conf = det
x1, y1, x2, y2 = bbox
print("detected:", phrase, "| conf:", round(conf, 4), "| bbox:", bbox)

sam2_predictor.set_image(image_np)
masks, scores, _ = sam2_predictor.predict(box=np.array([x1, y1, x2, y2]), multimask_output=True)
best_m = masks[int(np.argmax(scores))].astype(np.uint8) * 255
hard_mask, soft_mask = refine_mask(best_m, dilate_px=MASK_DILATE, feather_px=MASK_FEATHER)

vis = image_np.copy()
cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
overlay = vis.copy()
overlay[..., 1] = np.maximum(overlay[..., 1], hard_mask)
show_image(overlay, "Detected bbox + mask")
show_image(hard_mask, "Hard mask", cmap="gray")


In [ ]:
# @title 8) Generate replacement on ROI only
mb = mask_bbox(hard_mask)
if mb is None:
    raise RuntimeError("Mask is empty after refinement.")

rx1, ry1, rx2, ry2 = expand_bbox(mb, w, h, pad=ROI_PADDING)
roi_img = image_np[ry1:ry2, rx1:rx2]
roi_hard = hard_mask[ry1:ry2, rx1:rx2]
roi_soft = soft_mask[ry1:ry2, rx1:rx2]

show_image(roi_img, "ROI image")
show_image(roi_hard, "ROI hard mask", cmap="gray")

rw, rh = roi_img.shape[1], roi_img.shape[0]
rw, rh = clamp_size_keep_aspect(rw, rh, max_side=MAX_ROI_SIDE)
target_w = snap_to_multiple(rw, base=16, min_v=256)
target_h = snap_to_multiple(rh, base=16, min_v=256)

roi_img_pil = Image.fromarray(roi_img).resize((target_w, target_h), Image.Resampling.LANCZOS)
roi_mask_pil = Image.fromarray(roi_hard).resize((target_w, target_h), Image.Resampling.NEAREST)

prompt = (
    f"{NEW_OBJECT_PROMPT}. "
    "Photorealistic. Match original scene lighting, perspective, depth of field, color and noise. "
    "Replace only the masked object and keep surroundings consistent. "
    "If new object is smaller, reconstruct background naturally."
)

generator = torch.Generator(device="cpu").manual_seed(SEED)

sig = inspect.signature(pipe.__call__)
kwargs = {
    "prompt": prompt,
    "image": roi_img_pil,
    "mask_image": roi_mask_pil,
    "guidance_scale": GUIDANCE_SCALE,
    "num_inference_steps": NUM_STEPS,
    "generator": generator,
    "height": target_h,
    "width": target_w,
}
if "max_sequence_length" in sig.parameters:
    kwargs["max_sequence_length"] = 512
if "negative_prompt" in sig.parameters:
    kwargs["negative_prompt"] = NEGATIVE_PROMPT
if "true_cfg_scale" in sig.parameters:
    kwargs["true_cfg_scale"] = max(1.0, GUIDANCE_SCALE / 10.0)

try:
    with torch.inference_mode():
        gen_roi = pipe(**kwargs).images[0]
except torch.cuda.OutOfMemoryError:
    if DEVICE != "cuda":
        raise
    torch.cuda.empty_cache()
    print("OOM: retry with smaller ROI and fewer steps")
    small_w = max(256, (target_w * 3) // 4)
    small_h = max(256, (target_h * 3) // 4)
    small_w = snap_to_multiple(small_w, base=16, min_v=256)
    small_h = snap_to_multiple(small_h, base=16, min_v=256)
    kwargs["image"] = roi_img_pil.resize((small_w, small_h), Image.Resampling.LANCZOS)
    kwargs["mask_image"] = roi_mask_pil.resize((small_w, small_h), Image.Resampling.NEAREST)
    kwargs["width"] = small_w
    kwargs["height"] = small_h
    kwargs["num_inference_steps"] = max(10, NUM_STEPS // 2)
    with torch.inference_mode():
        gen_roi = pipe(**kwargs).images[0]

# back to original ROI size
gen_roi_np = np.array(gen_roi.resize((roi_img.shape[1], roi_img.shape[0]), Image.Resampling.LANCZOS))
show_image(gen_roi_np, f"Generated ROI ({quant_mode})")


In [ ]:
# @title 9) Composite and save
final_alpha = alpha_insert(image_np, gen_roi_np, roi_soft, rx1, ry1, rx2, ry2)
final_poisson = poisson_insert(image_np, gen_roi_np, roi_hard, rx1, ry1, rx2, ry2)

show_image(final_alpha, "Final (alpha blend)")
show_image(final_poisson, "Final (poisson blend)")

alpha_path = str(Path(OUT_DIR) / "output_replace_alpha.png")
poisson_path = str(Path(OUT_DIR) / "output_replace_poisson.png")
Image.fromarray(final_alpha).save(alpha_path)
Image.fromarray(final_poisson).save(poisson_path)
print("Saved:", alpha_path)
print("Saved:", poisson_path)


## Notes
- Если получаете `401` при загрузке FLUX модели: примите лицензию модели на Hugging Face и задайте `HF_TOKEN`.
- Если OOM сохраняется: уменьшите `MAX_ROI_SIDE` до `768`, `NUM_STEPS` до `12`, `GUIDANCE_SCALE` до `10`.
- Для сложной детекции снизьте `BOX_THRESHOLD/TEXT_THRESHOLD`.
